In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from aml_working_paper.data.preprocess import preprocess_data

In [20]:
EMISSION_COLUMNS = ["transaction_volume","cross_border_ratio","structuring_score"]

In [21]:
df_aml = pd.read_csv("/workspaces/aml-working-paper/aml_working_paper/data/synthetic_df_aml.csv")

In [22]:
account_ids = df_aml["account_id"].unique()
val_size_proportion = 0.20 / (0.20 + 0.10)

train_accounts, temp_accounts = train_test_split(account_ids, train_size=0.70, random_state=42, shuffle=True)
val_accounts, test_accounts = train_test_split(temp_accounts, train_size=val_size_proportion, random_state=42,shuffle=True)

train_df = df_aml[df_aml["account_id"].isin(train_accounts)].copy()
val_df = df_aml[df_aml["account_id"].isin(val_accounts)].copy()
test_df = df_aml[df_aml["account_id"].isin(test_accounts)].copy()

In [23]:
assert (train_df.shape[0] + val_df.shape[0] + test_df.shape[0] == df_aml.shape[0])

# Check that accounts do not overlap across splits
assert set(train_df["account_id"]).isdisjoint(val_df["account_id"])
assert set(train_df["account_id"]).isdisjoint(test_df["account_id"])
assert set(val_df["account_id"]).isdisjoint(test_df["account_id"])

In [24]:
print("Train AML-state distribution:")
print(train_df["aml_state"].value_counts(normalize=True).sort_index())

print("\nValidation AML-state distribution:")
print(val_df["aml_state"].value_counts(normalize=True).sort_index())

print("\nTest AML-state distribution:")
print(test_df["aml_state"].value_counts(normalize=True).sort_index())

Train AML-state distribution:
aml_state
0    0.482714
1    0.257000
2    0.260286
Name: proportion, dtype: float64

Validation AML-state distribution:
aml_state
0    0.4870
1    0.2385
2    0.2745
Name: proportion, dtype: float64

Test AML-state distribution:
aml_state
0    0.435
1    0.251
2    0.314
Name: proportion, dtype: float64


## Standarization 

In [25]:
scaler = StandardScaler()

In [ ]:

transform_df_train = train_df.copy()

features = transform_df_train[EMISSION_COLUMNS].copy()
features["transaction_volume"] = np.log1p(features["transaction_volume"])
scaled_features_train = scaler.fit_transform(features)
transform_df_train["transaction_volume_log"] = scaled_features_train[:, 0]
transform_df_train["cross_border_ratio_scaled"] = scaled_features_train[:, 1]
transform_df_train["structuring_score_scaled"] = scaled_features_train[:, 2]

In [39]:
transform_df_val = val_df.copy()

features = transform_df_val[EMISSION_COLUMNS].copy()
features["transaction_volume"] = np.log1p(features["transaction_volume"])
### not using fit_transform since I am using the same scaler from "fit_transform_train" 
scaled_features_val = scaler.transform(features) 
transform_df_val["transaction_volume_log"] = scaled_features_val[:, 0]
transform_df_val["cross_border_ratio_scaled"] = scaled_features_val[:, 1]
transform_df_val["structuring_score_scaled"] = scaled_features_val[:, 2]

In [ ]:
transform_df_test = val_df.copy()

features = transform_df_test[EMISSION_COLUMNS].copy()
features["transaction_volume"] = np.log1p(features["transaction_volume"])
### not using fit_transform since I am using the same scaler from "fit_transform_train" 
scaled_features_test = scaler.transform(features) 
transform_df_test["transaction_volume_log"] = scaled_features_test[:, 0]
transform_df_test["cross_border_ratio_scaled"] = scaled_features_test[:, 1]
transform_df_test["structuring_score_scaled"] = scaled_features_test[:, 2]

### Build Sequences

In [29]:
sequences_train = []
sequence_columns = ["transaction_volume_log", "cross_border_ratio_scaled","structuring_score_scaled"]

df_sorted_train = transform_df_train.sort_values(["account_id", "time"])

for _, account_df in df_sorted_train.groupby("account_id", sort=True):
    sequence = account_df[sequence_columns].to_numpy(dtype=np.float64)
    sequences_train.append(sequence)

In [30]:
sequences_val = []
sequence_columns = ["transaction_volume_log", "cross_border_ratio_scaled","structuring_score_scaled"]

df_sorted_val = transform_df_val.sort_values(["account_id", "time"])

for _, account_df in df_sorted_val.groupby("account_id", sort=True):
    sequence = account_df[sequence_columns].to_numpy(dtype=np.float64)
    sequences_val.append(sequence)

In [31]:
sequences_test = []
sequence_columns = ["transaction_volume_log", "cross_border_ratio_scaled","structuring_score_scaled"]

df_sorted_test = transform_df_test.sort_values(["account_id", "time"])

for _, account_df in df_sorted_test.groupby("account_id", sort=True):
    sequence = account_df[sequence_columns].to_numpy(dtype=np.float64)
    sequences_test.append(sequence)

## Using preprocessing.py

In [33]:
train_df, val_df, test_df, train_sequences, val_sequences, test_sequences, scaler = preprocess_data(df_aml)